[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nickmelamed/cal-nerds-research-agent/blob/main/notebooks/fullcode.ipynb)

# Research Paper Assistant (Build it Yourself)

This notebook starts with a small agent that already works. It has one
tool, `search_arxiv`, and a loop that asks the model, runs the tool it
picks, and repeats. It has no error handling, no limit on how long it runs,
and no record of how many tokens it uses. Those are your tasks.

The tasks are in priority order, so finishing two still gives you something
to share. Each task has a "Your turn" cell, a check, and a hint you can
open if you get stuck. The finished agent is in the demo notebook:
[open the demo](https://colab.research.google.com/github/nickmelamed/cal-nerds-research-agent/blob/main/notebooks/demo.ipynb).

Run the cells from top to bottom.

## 1. Setup

First we install the `openai` package. It is the client library, and it
also works with OpenRouter and Groq.

In [ ]:
!pip install -q openai

In [ ]:
# @title Helpers for talking to the model
"""Client setup for OpenRouter and Groq, 429 retry, and usage helpers."""

import os
import time
from getpass import getpass

import requests

MODEL = "openai/gpt-oss-120b"
MAX_429_RETRIES = 5
DEFAULT_WAIT_S = 2
MAX_WAIT_S = 30
TOKENS_HEADER = "x-ratelimit-remaining-tokens"

BASE_URLS = {
    "openrouter": "https://openrouter.ai/api/v1",
    "groq": "https://api.groq.com/openai/v1",
}
KEY_NAMES = {"openrouter": "OPENROUTER_API_KEY", "groq": "GROQ_API_KEY"}
OPENROUTER_KEY_URL = "https://openrouter.ai/api/v1/key"
GROQ_KEY_PREFIX = "gsk_"

_sleep = time.sleep
_last_headers: dict[str, str] = {}
_active: dict[str, str | None] = {"provider": None}


def _secret(name: str) -> str | None:
    key = os.environ.get(name)
    if key:
        return key
    try:
        from google.colab import userdata

        return userdata.get(name) or None
    except Exception:
        return None


def choose_provider() -> str:
    """Return ``"groq"`` when a student's own key is set, else ``"openrouter"``."""
    return "groq" if _secret(KEY_NAMES["groq"]) else "openrouter"


def get_api_key(provider: str | None = None) -> str:
    """Return the key for ``provider`` from the environment, Colab Secrets, or a prompt."""
    name = KEY_NAMES[provider or choose_provider()]
    return _secret(name) or getpass(f"{name}: ")


def make_client(provider: str | None = None):
    """Return an OpenAI-compatible client and remember which provider it uses.

    With no ``provider`` the choice follows ``choose_provider``. A key typed at
    the prompt that starts with ``gsk_`` switches to Groq.
    """
    from openai import OpenAI

    chosen = provider or choose_provider()
    key = _secret(KEY_NAMES[chosen]) or getpass(
        "Paste your API key (the workshop key, or your own Groq key): "
    )
    if provider is None and key.startswith(GROQ_KEY_PREFIX):
        chosen = "groq"
    _active["provider"] = chosen
    return OpenAI(base_url=BASE_URLS[chosen], api_key=key)


def active_provider() -> str | None:
    """Return the provider of the last client made, for the smoke test."""
    return _active["provider"]


def _retry_wait(error: Exception, attempt: int = 0) -> float:
    headers = getattr(getattr(error, "response", None), "headers", None) or {}
    try:
        wait = float(headers["retry-after"])
    except (KeyError, TypeError, ValueError):
        wait = DEFAULT_WAIT_S * 2**attempt
    return min(max(wait, 1), MAX_WAIT_S)


def chat(client, **kwargs):
    """Call the chat endpoint, retrying on 429.

    Response headers are kept for ``remaining_tokens``. After the retries run
    out the last 429 error is raised.
    """
    raw_api = client.chat.completions.with_raw_response
    for attempt in range(MAX_429_RETRIES + 1):
        try:
            raw = raw_api.create(**kwargs)
        except Exception as error:
            if getattr(error, "status_code", None) != 429 or attempt == MAX_429_RETRIES:
                raise
            wait = _retry_wait(error, attempt)
            print(f"The shared key is busy, retrying in {wait:g}s")
            _sleep(wait)
        else:
            _last_headers.clear()
            _last_headers.update({k.lower(): v for k, v in raw.headers.items()})
            return raw.parse()


def remaining_tokens() -> int | None:
    """Return the tokens left this minute from the last response, if reported.

    Only Groq sends this header.
    """
    try:
        return int(_last_headers[TOKENS_HEADER])
    except (KeyError, ValueError):
        return None


def openrouter_credit(key: str | None = None) -> float | None:
    """Return the credit left on the OpenRouter key in dollars, or None.

    None means the key has no limit set or the request failed.
    """
    key = key or _secret(KEY_NAMES["openrouter"])
    try:
        reply = requests.get(
            OPENROUTER_KEY_URL, headers={"Authorization": f"Bearer {key}"}, timeout=10
        )
        reply.raise_for_status()
        remaining = reply.json()["data"]["limit_remaining"]
    except Exception:
        return None
    return float(remaining) if remaining is not None else None


def smoke_test(client, model: str = MODEL) -> bool:
    """Send one tiny request and print whether the key and model work."""
    try:
        chat(client, model=model, messages=[{"role": "user", "content": "Say ready."}])
    except Exception as error:
        print(f"Smoke test failed with {active_provider()} ({type(error).__name__}).")
        print("Check that your key is right and has credit, then run this cell again.")
        return False
    print(f"Smoke test passed. Using {active_provider()} with model {model}.")
    return True

If you have your own Groq key in Colab Secrets as `GROQ_API_KEY`, the
notebook uses it. Otherwise it uses `OPENROUTER_API_KEY`, and if neither is
set it asks you to paste a key.

In [ ]:
client = make_client()
assert smoke_test(client), "Fix the key problem above, then run this cell again."

## 2. The Tool

`search_arxiv` sends a search to arXiv and turns the reply into a short
list of papers. arXiv asks for at most one request every 3 seconds, so the
function waits if it was called a moment ago.

In [ ]:
import json
import time
import xml.etree.ElementTree as ET

import requests

ARXIV_URL = "https://export.arxiv.org/api/query"
ATOM = {"a": "http://www.w3.org/2005/Atom"}
ARXIV_DELAY_S = 3

last_arxiv_call = 0.0


def search_arxiv(query, max_results=3):
    global last_arxiv_call
    wait = ARXIV_DELAY_S - (time.monotonic() - last_arxiv_call)
    if wait > 0:
        time.sleep(wait)
    last_arxiv_call = time.monotonic()

    params = {
        "search_query": "all:" + query,
        "max_results": min(int(max_results), 3),
        "sortBy": "relevance",
    }
    reply = requests.get(ARXIV_URL, params=params)
    root = ET.fromstring(reply.text)

    papers = []
    for entry in root.findall("a:entry", ATOM):
        full_id = entry.findtext("a:id", namespaces=ATOM)
        papers.append(
            {
                "id": full_id.split("/abs/")[-1],
                "title": " ".join(entry.findtext("a:title", namespaces=ATOM).split()),
                "date": entry.findtext("a:published", namespaces=ATOM)[:10],
                "link": full_id,
            }
        )
    return papers

Try it on its own before the agent uses it.

In [ ]:
search_arxiv("graph neural networks for molecules", 2)

## 3. Tell the Model What the Tool Does

The model never sees the Python above. It sees only this description, and
it decides whether to call the tool from the words in it.

In [ ]:
SEARCH_ARXIV_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_arxiv",
        "description": (
            "Search arXiv for research papers on a topic. Returns the id, "
            "title, date, and link of the best matches."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search terms"},
                "max_results": {
                    "type": "integer",
                    "description": "How many papers to return, 1 to 3",
                },
            },
            "required": ["query"],
        },
    },
}

TOOLS = [SEARCH_ARXIV_SCHEMA]

# A dictionary from a tool's name to the function that runs it.
TOOL_FUNCTIONS = {"search_arxiv": search_arxiv}

## 4. The Agent Loop

Each time around the loop we ask the model. If it asks for a tool, we run
the tool, add the result to the conversation, and ask again. When it
answers without asking for a tool, we are done.

`tool_results` keeps what every tool returned. The citation check at the
end uses it to see which papers the agent really retrieved.

In [ ]:
SYSTEM_PROMPT = (
    "You are a research paper assistant. Use search_arxiv to find papers. "
    "Search arXiv at most twice, then answer. arXiv search cannot filter by "
    "date, so do not put years in a query. Answer in a short paragraph and "
    "cite each paper by its arXiv id. Only cite papers your tools returned."
)

tool_results = []

`run_tool` runs one tool the model asked for. If the model asks for a tool
that does not exist or sends text that is not valid JSON, it returns a
message instead of stopping, so the model can fix its own mistake.

In [ ]:
def run_tool(name, raw_args):
    if name not in TOOL_FUNCTIONS:
        return f"There is no tool called {name}. Use one of: {', '.join(TOOL_FUNCTIONS)}."
    try:
        args = json.loads(raw_args or "{}")
    except ValueError:
        return "The tool arguments were not valid JSON. Try the call again."
    return TOOL_FUNCTIONS[name](**args)

In [ ]:
def run_agent(question):
    tool_results.clear()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    while True:
        response = chat(client, model=MODEL, messages=messages, tools=TOOLS)
        message = response.choices[0].message

        if not message.tool_calls:
            return message.content

        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": call.function.name,
                            "arguments": call.function.arguments,
                        },
                    }
                    for call in message.tool_calls
                ],
            }
        )
        for call in message.tool_calls:
            name = call.function.name
            result = run_tool(name, call.function.arguments)
            print(f"Tool call: {name}({call.function.arguments})")
            tool_results.append({"name": name, "args": call.function.arguments, "result": result})
            if not isinstance(result, str):
                result = json.dumps(result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

## 5. Ask a Question and Check the Citations

Models sometimes invent papers. The check below looks at every arXiv id in
the answer and counts it only if a tool returned it in this run. It checks
that a cited paper was retrieved. It does not check that what the answer
says about the paper is true.

In [ ]:
# @title Citation check
import re
from dataclasses import dataclass, field


def _normalize_id(paper_id):
    pid = paper_id.strip()
    pid = re.sub(r"^(https?://)?(www\.)?arxiv\.org/(abs|pdf)/", "", pid)
    pid = re.sub(r"^arxiv:", "", pid, flags=re.IGNORECASE)
    pid = re.sub(r"\.pdf$", "", pid)
    return re.sub(r"v\d+$", "", pid)


"""Check that every paper an answer cites was returned by a tool in this run."""

import re
from dataclasses import dataclass, field


NEW_ID = r"(?<![\d.])\d{4}\.\d{4,5}(?:v\d+)?(?![\d])"
OLD_ID = r"(?<![\w-])[a-z]+(?:-[a-z]+)*(?:\.[A-Z]{2})?/\d{7}(?:v\d+)?"
ID_PATTERN = re.compile(f"{NEW_ID}|{OLD_ID}")
# Models often write the hyphen of an id like quant-ph/0602157 as a Unicode
# hyphen or en dash, which would cut the id short at the first letter group.
UNICODE_DASHES = "\u2010\u2011\u2012\u2013\u2212\ufe63\uff0d"
LOOSE_OLD_ID = re.compile(
    rf"(?<![\w{UNICODE_DASHES}-])[a-z]+(?:[-{UNICODE_DASHES}][a-z]+)*"
    r"(?:\.[A-Z]{2})?/\d{7}(?:v\d+)?"
)
DASH_CHARS = re.compile(f"[{UNICODE_DASHES}]")
QUOTED = re.compile(r"[\"“]([^\"”]+)[\"”]")
MIN_QUOTED_WORDS = 4
# Text with these characters is an example or a formula, not a paper title.
NOT_TITLE_CHARS = re.compile(r"[\u2190-\u21ff=<>+*|{}\[\]^_~\\]")


@dataclass
class CitationReport:
    verified: list[str] = field(default_factory=list)
    unverified: list[str] = field(default_factory=list)
    summary: str = ""


def _looks_like_title(phrase: str) -> bool:
    phrase = phrase.strip()
    return (
        bool(phrase)
        and phrase[0].isupper()
        and not phrase.endswith(".")
        and not NOT_TITLE_CHARS.search(phrase)
    )


def _norm_title(text: str) -> str:
    return " ".join(re.findall(r"[a-z0-9]+", text.lower()))


def _retrieved(tool_results: list[dict]) -> tuple[dict[str, str], set[str]]:
    # Wikipedia titles go in the title set only, so a quoted one is not flagged.
    papers: dict[str, str] = {}
    titles: set[str] = set()
    for entry in tool_results:
        result = entry["result"]
        items = result if isinstance(result, list) else [result]
        for item in items:
            if not isinstance(item, dict) or "title" not in item:
                continue
            title = _norm_title(item["title"])
            titles.add(title)
            if entry["name"] != "lookup_wikipedia" and "id" in item:
                papers[_normalize_id(item["id"])] = title
    return papers, titles


def check_citations(answer: str, tool_results: list[dict]) -> CitationReport:
    """Count a citation as verified only if a tool returned it in this run.

    Match papers by arXiv id, or by title when the answer gives no id. A
    quoted phrase of four or more words counts as a reference to any
    retrieved paper whose title contains it. It is unverified when no
    retrieved title does, unless it reads as an example rather than a title:
    it starts in lowercase, ends in a period, or has arrows or math symbols.
    """
    answer = LOOSE_OLD_ID.sub(lambda m: DASH_CHARS.sub("-", m.group()), answer)
    papers, titles = _retrieved(tool_results)
    report = CitationReport()
    seen_ids: set[str] = set()

    for match in ID_PATTERN.findall(answer):
        pid = _normalize_id(match)
        if pid in seen_ids:
            continue
        seen_ids.add(pid)
        (report.verified if pid in papers else report.unverified).append(pid)

    padded = f" {_norm_title(answer)} "
    for pid, title in papers.items():
        if pid not in seen_ids and f" {title} " in padded:
            seen_ids.add(pid)
            report.verified.append(pid)

    for quoted in QUOTED.findall(answer):
        title = _norm_title(quoted)
        if len(title.split()) < MIN_QUOTED_WORDS:
            continue
        needle = f" {title} "
        matched = [pid for pid, known in papers.items() if needle in f" {known} "]
        if matched:
            for pid in matched:
                if pid not in seen_ids:
                    seen_ids.add(pid)
                    report.verified.append(pid)
        elif _looks_like_title(quoted) and not any(needle in f" {known} " for known in titles):
            report.unverified.append(quoted.strip())

    total = len(report.verified) + len(report.unverified)
    if total == 0:
        report.summary = "The answer cites no papers."
    else:
        report.summary = (
            f"{len(report.verified)} of {total} cited papers "
            "were retrieved by the agent in this run."
        )
    return report

In [ ]:
QUESTION = (
    "Find a few papers on machine learning for weather forecasting and "
    "explain in plain language what one of them does."
)

answer = run_agent(QUESTION)
print()
print(answer)

In [ ]:
report = check_citations(answer, tool_results)
print(report.summary)
if report.unverified:
    print("Not retrieved by any tool:", report.unverified)

## Task 1: Write a New Tool

Give the agent a second tool. Pick one from the options below (or both if
you're feeling adventurous!):

- `save_to_reading_list(paper_id, note)` stores a paper and a short note in
  a list, and returns a message saying it was saved.
- A date filter for `search_arxiv`, so the agent can ask for papers from a
  given year or later. arXiv cannot filter by date itself, so you will
  filter the results after they come back.

You need three things. The function, its schema (the description the model
reads), and a line in `TOOL_FUNCTIONS` and `TOOLS` so the loop knows about
it.

**Check:** ask a question where the new tool is useful, and see whether the
agent uses it without being told. If it does not, read your description
again. The model decides from those words alone.

<details>
<summary>Hint!</summary>

Write the function first and call it by hand to see it works. Then copy
`SEARCH_ARXIV_SCHEMA`, change the name, the description, and the
parameters. Add the schema to `TOOLS` and the function to `TOOL_FUNCTIONS`
under the same name as in the schema.

</details>

In [ ]:
# Your turn: write the new tool and its schema here.

In [ ]:
# Add your schema to TOOLS and your function to TOOL_FUNCTIONS, then ask a
# question that should use it.
QUESTION = "Find two papers on protein folding and save the best one to my reading list."

answer = run_agent(QUESTION)
print()
print(answer)

## Task 2: Handle Errors

The loop already answers the model when it asks for a tool that does not
exist or sends text that is not valid JSON. Anything else that goes wrong
still stops the whole run with a red error. Make the agent keep going
instead. Handle these three cases:

- A search that finds nothing. Return a message the model can act on, such
  as "No papers found, try broader terms".
- A network failure or a slow reply. Give `requests.get` a `timeout`, and
  catch the error it raises.
- A tool that raises an error, or arguments that do not fit the function,
  such as a missing `query`. Send a message back as the tool result.

The pattern is the same each time. Catch the problem and return a short
message that tells the model what happened, so it can try something else.

**Check:** run the test cell below. It needs no internet.

<details>
<summary>Hint!</summary>

Wrap the risky lines in `try:` and `except SomeError:`. In the search
function that is the `requests.get` call and the XML parsing. In `run_tool`
it is the line that calls the tool. `requests.RequestException` covers
timeouts and other network errors. Check `if not papers:` before returning.

</details>

In [ ]:
# Your turn: add error handling to search_arxiv and run_agent above, then
# run the test cell below.

In [ ]:
# @title Test for task 2
def test_error_handling():
    original_get = requests.get
    problems = []

    def fake_timeout(*args, **kwargs):
        raise requests.Timeout("forced timeout")

    def broken_tool(**kwargs):
        raise RuntimeError("the tool broke")

    cases = [
        ("empty query", lambda: search_arxiv("")),
        ("forced timeout", lambda: search_arxiv("graph neural networks")),
        ("bad arguments", lambda: run_tool("search_arxiv", '{"wrong_name": 1}')),
        ("tool that raises", lambda: run_tool("broken_tool", "{}")),
    ]

    # Every request fails, so nothing here touches the internet.
    requests.get = fake_timeout
    TOOL_FUNCTIONS["broken_tool"] = broken_tool
    try:
        for label, call in cases:
            try:
                outcome = call()
                if not isinstance(outcome, (str, list)):
                    problems.append(f"{label}: expected a message or a list")
            except Exception as error:
                problems.append(f"{label}: raised {type(error).__name__}")
    finally:
        requests.get = original_get
        del TOOL_FUNCTIONS["broken_tool"]

    if problems:
        print("Not yet:")
        for problem in problems:
            print(" -", problem)
    else:
        print("Passed: the tools return messages instead of raising.")


test_error_handling()

## Task 3: Cap the Loop

`while True` never stops if the model keeps asking for tools. Add a
`max_steps` argument to `run_agent`, so `run_agent(question, max_steps=6)`
makes at most six model calls. If the cap is reached, return a short
message instead of an answer, such as "Stopped after 6 steps."

**Check:** run the test cell below. It uses a pretend model that asks for a
tool every time, so it needs no key.

<details>
<summary>Hint!</summary>

Replace `while True:` with `for step in range(max_steps):`. If the loop
ends without returning, the cap was reached, so return your message after
the loop. Give `max_steps` a default value so older calls still work.

</details>

In [ ]:
# Your turn: add max_steps to run_agent above, then run the test cell below.

In [ ]:
# @title Test for task 3
from types import SimpleNamespace


class AlwaysSearches:
    """A pretend model that asks for a search on every call."""

    def __init__(self):
        call = SimpleNamespace(
            id="call-1",
            function=SimpleNamespace(
                name="search_arxiv", arguments='{"query": "test"}'
            ),
        )
        message = SimpleNamespace(content="", tool_calls=[call])
        reply = SimpleNamespace(
            choices=[SimpleNamespace(message=message)],
            usage=SimpleNamespace(total_tokens=10),
        )
        self.chat = self
        self.completions = self
        self.with_raw_response = self
        self.calls = 0
        self._reply = SimpleNamespace(headers={}, parse=lambda: reply)

    def create(self, **kwargs):
        self.calls += 1
        return self._reply


def test_step_cap():
    global client
    real_client, real_search = client, search_arxiv
    pretend = AlwaysSearches()
    client = pretend
    TOOL_FUNCTIONS["search_arxiv"] = lambda **kwargs: []
    try:
        outcome = run_agent("Any question", max_steps=1)
    except Exception as error:
        print("Not yet:", type(error).__name__, error)
        return
    finally:
        client = real_client
        TOOL_FUNCTIONS["search_arxiv"] = real_search

    if pretend.calls != 1:
        print(f"Not yet: the model was called {pretend.calls} times, expected 1.")
    elif not isinstance(outcome, str) or not outcome:
        print("Not yet: run_agent should return a message at the cap.")
    else:
        print("Passed: the loop stopped after 1 step and returned:", outcome)


test_step_cap()

## Task 4: Log Tokens and Time

Every model call costs tokens and takes time.

Record both for each call, then print a summary at the end of a run. A
table like this is enough:

```
step  tokens  seconds
1     812     1.4
2     1650    2.1
total 2462    3.5
```

Each response has `response.usage.total_tokens`. Time the call with
`time.monotonic()` before and after it.

**Check:** run a question and see that the summary prints.

<details>
<summary>Hint!</summary>

Make an empty list called `call_log` at the top of `run_agent`. After each
`chat(...)` call, append a dictionary with the step number, the tokens,
and the seconds. After the loop ends, print one line per entry and one line
with the totals. Some providers leave `usage` empty, so use `0` when it is
missing.

</details>

In [ ]:
# Your turn: add logging to run_agent above, then run a question below.
QUESTION = (
    "Find a few papers on machine learning for weather forecasting and "
    "explain in plain language what one of them does."
)

answer = run_agent(QUESTION)
print()
print(answer)

## Go Further

If you finish early, you can also try these implementations:

- Add `get_abstract` and `lookup_wikipedia` as new tools, written from
  scratch. The demo notebook has working versions to compare against once
  you have tried.
- Make `check_citations` stricter. Right now it checks arXiv ids. Make it
  also match the titles of papers the agent mentions.